In [1]:
2+2

4

In [2]:
import os
import re
import uuid
import json
from typing import List, Dict
from dataclasses import dataclass, asdict

# PDF / DOCX
import PyPDF2
import docx

# Mistral (for embeddings)
from mistralai import Mistral

# Pinecone
from pinecone import Pinecone, ServerlessSpec


# =========================
# CONFIG
# =========================
OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
MISTRAL_API_KEY  = os.getenv("MISTRAL_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_CLOUD   = "aws"
PINECONE_REGION  = "us-east-1"

EMBED_MODEL = "mistral-embed"   # 1024-dimensional vectors
Tool_MODEL   = "openai/gpt-oss-120b"  
CHAT_MODEL   = "llama-3.3-70b-versatile" 
MEMORY_FILE = "system_memory.json"



mistral_client = Mistral(api_key=MISTRAL_API_KEY)
pc= Pinecone(api_key=PINECONE_API_KEY)

In [3]:
from mistralai import Mistral
import os
MISTRAL_API_KEY  = os.getenv("MISTRAL_API_KEY")
mistral_client = Mistral(api_key=MISTRAL_API_KEY)
EMBED_MODEL = "mistral-embed"   # 1024-dimensional vectors


In [4]:
from groq import Groq
from pydantic import BaseModel

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
Tool_MODEL   = "openai/gpt-oss-120b"  
CHAT_MODEL   = "llama-3.3-70b-versatile"  
Small_MODEL  = "llama-3.1-8b-instant"      
groq_client = Groq(api_key=GROQ_API_KEY)

In [5]:
# =========================
# DATA STRUCTURES
# =========================
@dataclass
class DocumentAgent:
    doc_id: str
    summary: str
    keywords: List[str]
    vector_db: str
    metadata: Dict
    # ↑ metadata will now also carry:
    #   - file_name   → original file (e.g. "transformers_paper.pdf")
    #   - embed_model → which model was used
    #   - type / status



In [6]:

# =========================
# STEP 1: EXTRACT TEXT
# =========================
def extract_text(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        text = []
        with open(file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text.append(page.extract_text() or "")
        return "\n".join(text)

    elif ext == ".docx":
        doc = docx.Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    else:
        raise ValueError("Unsupported file format. Use PDF, DOCX, or TXT.")


In [7]:
# =========================
# STEP 2: CLEAN + NORMALIZE
# =========================
def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s.,;:!?()-]", "", text)
    return text.strip()


# =========================
# STEP 3: CHUNK TEXT
# =========================
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    words = text.split()

    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

        if chunk_size <= overlap:
            break

    return chunks


In [8]:
# =========================
# STEP 4: GENERATE EMBEDDINGS  ← Mistral
# =========================
def generate_embeddings(chunks: List[str]) -> List[List[float]]:
    response = mistral_client.embeddings.create(
        model=EMBED_MODEL,
        inputs=chunks          # batched — single API call
    )
    return [item.embedding for item in response.data]



In [9]:
# =========================
# STEP 5: STORE IN PINECONE
# =========================
def store_in_pinecone(doc_id: str, chunks: List[str], embeddings: List[List[float]]) -> str:
    index_name = f"{doc_id.lower()}-index"

    existing_indexes = [idx["name"] for idx in pc.list_indexes()]

    if index_name not in existing_indexes:
        pc.create_index(
            name=index_name,
            dimension=1024,        # mistral-embed is always 1024-d
            metric="cosine",
            spec=ServerlessSpec(
                cloud=PINECONE_CLOUD,
                region=PINECONE_REGION
            )
        )

    index = pc.Index(index_name)

    vectors = []
    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        vectors.append({
            "id": f"{doc_id}_chunk_{i}",
            "values": emb,
            "metadata": {
                "text": chunk,
                "doc_id": doc_id,
                "chunk_id": i
            }
        })

    index.upsert(vectors=vectors)
    return index_name


### Checking for rate limits and implementing a retry mechanism with exponential backoff is crucial when working with APIs that have strict usage limits, such as Groq's free tier. Below is an implementation of a helper function that wraps around the Groq API calls to handle rate limits gracefully.


In [10]:
import time
from groq import Groq, RateLimitError, APIStatusError
# Groq free tier TPM limit per model
GROQ_TPM_LIMIT  = 131072
GROQ_TPM_BUFFER = 0.85    # pause at 85% of limit to be safe
TOKEN_WINDOW    = 60      # Groq resets token count every 60 seconds

# Track usage across calls
_token_tracker = {
    "tokens_used": 0,
    "window_start": time.time()
}


def _reset_token_window_if_needed():
    """Reset token counter if 60s window has passed."""
    now = time.time()
    elapsed = now - _token_tracker["window_start"]

    if elapsed >= TOKEN_WINDOW:
        print(f"[INFO] Token window reset. ({elapsed:.1f}s elapsed)")
        _token_tracker["tokens_used"] = 0
        _token_tracker["window_start"] = now


def _wait_if_token_limit_near(tokens_about_to_use: int):
    """
    If adding the next call's tokens would exceed 85% of TPM limit,
    sleep until the 60s window resets.
    """
    _reset_token_window_if_needed()

    projected = _token_tracker["tokens_used"] + tokens_about_to_use
    limit      = GROQ_TPM_LIMIT * GROQ_TPM_BUFFER

    if projected >= limit:
        elapsed   = time.time() - _token_tracker["window_start"]
        wait_time = max(0, TOKEN_WINDOW - elapsed + 2)   # +2s buffer
        print(f"[RATE LIMIT] Token budget near limit ({_token_tracker['tokens_used']:,} used). "
              f"Waiting {wait_time:.1f}s for window reset...")
        time.sleep(wait_time)

        # Reset after waiting
        _token_tracker["tokens_used"] = 0
        _token_tracker["window_start"] = time.time()


def _call_groq_with_retry(messages, model, response_format=None, max_retries=5):
    """
    Groq call with:
    - Proactive token tracking (pauses before hitting limit)
    - Retry + exponential backoff as safety net
    """
    # Estimate tokens about to be used (rough: 1 token ≈ 4 chars)
    estimated_tokens = sum(len(m["content"]) // 4 for m in messages)
    _wait_if_token_limit_near(estimated_tokens)

    delay = 5
    for attempt in range(max_retries):
        try:
            kwargs = {"model": model, "messages": messages}
            if response_format:
                kwargs["response_format"] = response_format

            response = groq_client.chat.completions.create(**kwargs)

            # Track actual tokens used from response
            actual_tokens = response.usage.total_tokens
            _token_tracker["tokens_used"] += actual_tokens
            print(f"[TOKEN] Used {actual_tokens} tokens this call | "
                  f"Total this window: {_token_tracker['tokens_used']:,}/{GROQ_TPM_LIMIT:,}")

            return response

        except RateLimitError as e:
            if attempt == max_retries - 1:
                raise

            wait_time = _parse_retry_after(str(e), fallback=delay)
            print(f"[RATE LIMIT] Hit on attempt {attempt+1}. Waiting {wait_time}s...")
            time.sleep(wait_time)

            # Reset window after waiting
            _token_tracker["tokens_used"] = 0
            _token_tracker["window_start"] = time.time()
            delay *= 2


def _parse_retry_after(error_msg: str, fallback: int) -> int:
    match = re.search(r"(?:try again in|retry after)\s*([\d.]+)", error_msg, re.IGNORECASE)
    if match:
        return int(float(match.group(1))) + 1
    return fallback

In [11]:
from pydantic import BaseModel

class DocumentSummary(BaseModel):
    summary: str
    keywords: List[str]

summarize_llm="meta-llama/llama-4-scout-17b-16e-instruct"


def _summarize_chunk(chunk: str, chunk_index: int, total_chunks: int) -> str:
    response = _call_groq_with_retry(
        model=summarize_llm,
        messages=[
            {"role": "system", "content": "You are a document analysis system. Summarize the given text concisely in max 100 words."},
            {"role": "user",   "content": f"Chunk {chunk_index+1} of {total_chunks}:\n\n{chunk}"}
        ]
    )
    return response.choices[0].message.content.strip()


def _merge_summaries(chunk_summaries: List[str]) -> DocumentSummary:
    combined = "\n\n".join(f"[Part {i+1}]: {s}" for i, s in enumerate(chunk_summaries))

    response = _call_groq_with_retry(
        model=summarize_llm,
        messages=[
            {"role": "system", "content": "You are a document analysis system."},
            {"role": "user",   "content": f"""
These are summaries of different parts of a single document.
Produce:
1. One final concise summary (max 50 words) of the whole document
2. 5 most important keywords across all parts

Parts:
{combined}
"""}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name":   "DocumentSummary",
                "schema": DocumentSummary.model_json_schema()
            }
        }
    )
    return DocumentSummary.model_validate_json(response.choices[0].message.content)

In [12]:
# =========================
# STEP 6: SUMMARY + KEYWORDS 
# =========================

def summarize_and_extract_keywords(text: str) -> dict:
    CHUNK_SIZE = 12000

    if len(text) <= CHUNK_SIZE:
        chunks = [text]
    else:
        chunks = [text[i:i+CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE - 500)]
        print(f"[INFO] Document split into {len(chunks)} chunks for summarization.")

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"[INFO] Summarizing chunk {i+1}/{len(chunks)}...")
        # ↑ _call_groq_with_retry inside will auto-pause if token limit near
        summary = _summarize_chunk(chunk, i, len(chunks))
        chunk_summaries.append(summary)

    print("[INFO] Merging chunk summaries...")
    result = _merge_summaries(chunk_summaries)
    return {"summary": result.summary, "keywords": result.keywords}

In [13]:
# =========================
# STEP 7: CREATE DOCUMENT AGENT
# ← CHANGED: file_name now stored in metadata
#   so the retrieval router can show which file each doc came from
# =========================
def create_document_agent(
    doc_id: str,
    summary_data: Dict,
    vector_db: str,
    file_name: str          # ← new param
) -> DocumentAgent:
    return DocumentAgent(
        doc_id=doc_id,
        summary=summary_data["summary"],
        keywords=summary_data["keywords"],
        vector_db=vector_db,
        metadata={
            "type":        "document_agent",
            "status":      "active",
            "embed_model": EMBED_MODEL,
            "file_name":   file_name,   # ← e.g. "transformers_paper.pdf"
        }
    )


In [14]:
# =========================
# STEP 8: REGISTER IN SYSTEM MEMORY
# =========================
def register_document_agent(agent: DocumentAgent):
    try:
        with open(MEMORY_FILE, "r") as f:
            memory = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        memory = {}

    memory[agent.doc_id] = asdict(agent)

    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)

    print(f"[INFO] Agent '{agent.doc_id}' saved to {MEMORY_FILE}")

def load_system_memory() -> Dict:
    try:
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

In [15]:
# =========================
# MAIN PIPELINE
# =========================
def upload_document_pipeline(file_path: str) -> DocumentAgent:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        doc_id    = f"doc-{uuid.uuid4().hex[:8]}"
        file_name = os.path.basename(file_path)   # ← extract just filename

        print(f"[INFO] Starting pipeline for '{file_name}' → doc_id: {doc_id}")

        print("[INFO] Extracting text...")
        raw_text = extract_text(file_path)

        print("[INFO] Cleaning text...")
        cleaned_text = clean_text(raw_text)

        if not cleaned_text:
            raise ValueError("Document appears to be empty after cleaning.")

        print("[INFO] Chunking text...")
        chunks = chunk_text(cleaned_text)
        print(f"[INFO] Created {len(chunks)} chunks.")

        print("[INFO] Generating embeddings with Mistral...")
        embeddings = generate_embeddings(chunks)

        print("[INFO] Storing in Pinecone...")
        vector_db = store_in_pinecone(doc_id, chunks, embeddings)

        print("[INFO] Summarizing document...")
        summary_data = summarize_and_extract_keywords(cleaned_text)

        # ← file_name passed into agent creation
        agent = create_document_agent(doc_id, summary_data, vector_db, file_name)
        register_document_agent(agent)

        print("[INFO] Pipeline complete.")
        return agent

    except Exception as e:
        print(f"[ERROR] Pipeline failed for '{file_path}': {e}")
        raise

In [31]:
# =========================
# EXAMPLE USAGE
# =========================
if __name__ == "__main__":
    file_path = "/home/anish/Documents/Ai agent/data/pdf/paper.pdf"

    agent = upload_document_pipeline(file_path)

    print("\nDocument Agent Created:")
    print(f"  doc_id     : {agent.doc_id}")
    print(f"  file_name  : {agent.metadata['file_name']}")
    print(f"  summary    : {agent.summary}")
    print(f"  keywords   : {agent.keywords}")
    print(f"  vector DB  : {agent.vector_db}")
    print(f"  embed model: {agent.metadata['embed_model']}")

[INFO] Starting pipeline for 'paper.pdf' → doc_id: doc-2fd8f544
[INFO] Extracting text...
[INFO] Cleaning text...
[INFO] Chunking text...
[INFO] Created 30 chunks.
[INFO] Generating embeddings with Mistral...
[INFO] Storing in Pinecone...
[INFO] Summarizing document...
[INFO] Document split into 9 chunks for summarization.
[INFO] Summarizing chunk 1/9...
[INFO] Token window reset. (293.5s elapsed)
[TOKEN] Used 2724 tokens this call | Total this window: 2,724/131,072
[INFO] Summarizing chunk 2/9...
[TOKEN] Used 2510 tokens this call | Total this window: 5,234/131,072
[INFO] Summarizing chunk 3/9...
[TOKEN] Used 3624 tokens this call | Total this window: 8,858/131,072
[INFO] Summarizing chunk 4/9...
[TOKEN] Used 2994 tokens this call | Total this window: 11,852/131,072
[INFO] Summarizing chunk 5/9...
[TOKEN] Used 2765 tokens this call | Total this window: 14,617/131,072
[INFO] Summarizing chunk 6/9...
[TOKEN] Used 2647 tokens this call | Total this window: 17,264/131,072
[INFO] Summarizi

In [33]:
print(agent)

DocumentAgent(doc_id='doc-2fd8f544', summary='AIRA 2 is an AI research agent that overcomes bottlenecks in AI research, achieving state-of-the-art results on complex machine learning tasks. It uses asynchronous multi-GPU execution, evolutionary search, and hidden consistent evaluation to enable dynamic scoping and interactive debugging.', keywords=['AIRA 2', 'AI research agent', 'asynchronous multi-GPU execution', 'evolutionary search', 'hidden consistent evaluation'], vector_db='doc-2fd8f544-index', metadata={'type': 'document_agent', 'status': 'active', 'embed_model': 'mistral-embed', 'file_name': 'paper.pdf'})


In [25]:
data=load_system_memory()
print(data)

{'doc-2fd8f544': {'doc_id': 'doc-2fd8f544', 'summary': 'AIRA 2 is an AI research agent that overcomes bottlenecks in AI research, achieving state-of-the-art results on complex machine learning tasks. It uses asynchronous multi-GPU execution, evolutionary search, and hidden consistent evaluation to enable dynamic scoping and interactive debugging.', 'keywords': ['AIRA 2', 'AI research agent', 'asynchronous multi-GPU execution', 'evolutionary search', 'hidden consistent evaluation'], 'vector_db': 'doc-2fd8f544-index', 'metadata': {'type': 'document_agent', 'status': 'active', 'embed_model': 'mistral-embed', 'file_name': 'paper.pdf'}}, 'doc-38599d31': {'doc_id': 'doc-38599d31', 'summary': 'The document outlines a comprehensive learning path for mastering automation and AI-powered workflows, launching a successful automation agency, and a 4-step sequence for offering high-ticket services. It covers tools, AI agent architecture, and business growth phases.', 'keywords': ['automation', 'AI',

In [26]:
def list_available_documents() -> None:
    memory = load_system_memory()
    if not memory:
        print("[INFO] No documents registered yet.")
        return
    print(f"[INFO] {len(memory)} document(s) found in system memory:")
    print(f"\n{'='*55}")
    for doc_id, data in memory.items():
        print(f"  doc_id  : {doc_id}")
        print(f"  summary : {data['summary']}")
        print(f"  keywords: {', '.join(data['keywords'])}")
        print(f"  index   : {data['vector_db']}")
        print(f"  {'-'*53}")


In [27]:
list_available_documents()

[INFO] 2 document(s) found in system memory:

  doc_id  : doc-2fd8f544
  summary : AIRA 2 is an AI research agent that overcomes bottlenecks in AI research, achieving state-of-the-art results on complex machine learning tasks. It uses asynchronous multi-GPU execution, evolutionary search, and hidden consistent evaluation to enable dynamic scoping and interactive debugging.
  keywords: AIRA 2, AI research agent, asynchronous multi-GPU execution, evolutionary search, hidden consistent evaluation
  index   : doc-2fd8f544-index
  -----------------------------------------------------
  doc_id  : doc-38599d31
  summary : The document outlines a comprehensive learning path for mastering automation and AI-powered workflows, launching a successful automation agency, and a 4-step sequence for offering high-ticket services. It covers tools, AI agent architecture, and business growth phases.
  keywords: automation, AI, workflows, agency, business growth
  index   : doc-38599d31-index
  -----------

# Retrive code

In [17]:
from pydantic import BaseModel

class RouteDecision(BaseModel):
    doc_ids: List[str]   # ← enforces it's always a list of strings


def route_query_to_documents(query: str, memory: Dict) -> List[str]:
    if not memory:
        raise RuntimeError("No documents in memory.")

    doc_catalog = []
    for doc_id, data in memory.items():
        doc_catalog.append(
            f"doc_id: {doc_id}\n"
            f"file: {data['metadata'].get('file_name', 'unknown')}\n"
            f"summary: {data['summary']}\n"
            f"keywords: {', '.join(data['keywords'])}"
        )

    catalog_text = "\n\n".join(doc_catalog)

    prompt = f"""
You are a document routing agent. Given a user query, select the
doc_ids of documents that are likely to contain the answer.
Return an empty list if nothing is relevant.

--- AVAILABLE DOCUMENTS ---
{catalog_text}
--- END OF DOCUMENTS ---

User Query: {query}
"""

    response = groq_client.chat.completions.create(
        model=Tool_MODEL,
        messages=[
            {"role": "system", "content": "You are a document routing agent."},
            {"role": "user",   "content": prompt}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name":   "RouteDecision",
                "schema": RouteDecision.model_json_schema()  # ← { doc_ids: [...] }
            }
        }
    )

    content = response.choices[0].message.content
    result  = RouteDecision.model_validate_json(content)  # ← validated, no regex

    # Filter to only doc_ids that actually exist in memory
    valid_doc_ids = [d for d in result.doc_ids if d in memory]

    print(f"[ROUTER] Query routed to {len(valid_doc_ids)} document(s): {valid_doc_ids}")
    return valid_doc_ids

In [18]:
# =========================
# STEP 2: EMBED QUERY  ← Mistral
# =========================
def embed_query(query: str) -> List[float]:
    response = mistral_client.embeddings.create(
        model=EMBED_MODEL,
        inputs=[query]
    )
    return response.data[0].embedding

In [19]:
# =========================
# STEP 3: SEARCH A SINGLE PINECONE INDEX
# index_name comes from memory[doc_id]["vector_db"]
# =========================
def search_index(index_name: str, query_embedding: List[float], top_k: int = 5) -> List[Dict]:
    index = pc.Index(index_name)

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    matches = []
    for match in results["matches"]:
        matches.append({
            "score":    round(match["score"], 4),
            "text":     match["metadata"]["text"],
            "doc_id":   match["metadata"]["doc_id"],
            "chunk_id": match["metadata"]["chunk_id"],
        })

    return matches

In [20]:
# =========================
# STEP 4: SMART SEARCH
# Router decides which indexes to use, then searches only those
# =========================
def smart_search(query: str, top_k: int = 5) -> List[Dict]:
    """
    1. Load memory (summaries + keywords + index names)
    2. LLM router picks relevant doc_ids
    3. Embed the query once
    4. Search only the selected indexes
    5. Merge + rank results
    """
    memory = load_system_memory()

    # Step 1: Agent decides which documents to search
    relevant_doc_ids = route_query_to_documents(query, memory)

    if not relevant_doc_ids:
        print("[ROUTER] No relevant documents found for this query.")
        return []

    # Step 2: Embed query (single Mistral call)
    query_embedding = embed_query(query)

    # Step 3: Search only the relevant indexes
    all_matches = []
    for doc_id in relevant_doc_ids:
        index_name = memory[doc_id]["vector_db"]   # ← this is how index_name is obtained
        print(f"[INFO] Searching index '{index_name}' for doc '{doc_id}' ...")

        try:
            matches = search_index(index_name, query_embedding, top_k=top_k)
            all_matches.extend(matches)
        except Exception as e:
            print(f"[WARNING] Failed to search index '{index_name}': {e}")

    # Step 4: Rank all results across selected docs by score
    all_matches.sort(key=lambda x: x["score"], reverse=True)
    return all_matches[:top_k]

In [21]:


# =========================
# STEP 5: BUILD CONTEXT
# =========================
def build_context(matches: List[Dict]) -> str:
    parts = []
    for i, match in enumerate(matches, 1):
        parts.append(
            f"[Chunk {i} | doc: {match['doc_id']} | score: {match['score']}]\n{match['text']}"
        )
    return "\n\n---\n\n".join(parts)

In [22]:

# =========================
# STEP 6: ANSWER WITH LLM
# =========================
def answer_query(query: str, context: str) -> str:
    prompt = f"""
You are a helpful assistant. Answer the user's question using ONLY
the document context below. If the context does not contain enough
information, say No clearly.

--- DOCUMENT CONTEXT ---
{context}
--- END CONTEXT ---

User Question: {query}

Answer:
"""
    response = groq_client.chat.completions.create(
        model=Tool_MODEL,
        messages=[
            {"role": "system", "content": "You are a document Q&A assistant. Answer only from the provided context."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content.strip()

In [23]:

# =========================
# MAIN RETRIEVAL FUNCTION
# =========================
def retrieve_and_answer(
    query: str,
    top_k: int = 5,
    show_chunks: bool = False
) -> str:
    """
    Full smart retrieval pipeline:
      1. Agent reads summaries/keywords → picks relevant docs
      2. Embeds query with Mistral
      3. Searches only the relevant Pinecone indexes
      4. LLM answers from retrieved context
    """
    print(f"\n[QUERY] {query}")

    matches = smart_search(query, top_k=top_k)

    if not matches:
        return "No relevant content found for your query."

    if show_chunks:
        print("\n--- Retrieved Chunks ---")
        for m in matches:
            print(f"  [score: {m['score']} | doc: {m['doc_id']}] {m['text'][:120]}...")
        print("------------------------\n")

    context = build_context(matches)
    return answer_query(query, context)

In [38]:

# =========================
# EXAMPLE USAGE
# =========================
if __name__ == "__main__":

    list_available_documents()

    answer = retrieve_and_answer(
        query="How does AIRA 2 work?",
        top_k=5,
        show_chunks=True
    )
    print(f"\n[ANSWER]\n{answer}")

[INFO] 1 document(s) found in system memory:

  doc_id  : doc-2fd8f544
  summary : AIRA 2 is an AI research agent that overcomes bottlenecks in AI research, achieving state-of-the-art results on complex machine learning tasks. It uses asynchronous multi-GPU execution, evolutionary search, and hidden consistent evaluation to enable dynamic scoping and interactive debugging.
  keywords: AIRA 2, AI research agent, asynchronous multi-GPU execution, evolutionary search, hidden consistent evaluation
  index   : doc-2fd8f544-index
  -----------------------------------------------------

[QUERY] How does AIRA 2 work?
[ROUTER] Query routed to 1 document(s): ['doc-2fd8f544']
[INFO] Searching index 'doc-2fd8f544-index' for doc 'doc-2fd8f544' ...


/home/anish/Documents/Ai agent/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- Retrieved Chunks ---
  [score: 0.799 | doc: doc-2fd8f544] AIRA 2: Overcoming Bottlenecks in AI Research Agents Karen Hambardzumyan1,2,,Nicolas Baldwin1,,Edan Toledo1,2,,Rishi Haz...
  [score: 0.7843 | doc: doc-2fd8f544] Resource Allocation.We enforce a strict 1:1 worker-to-GPU mapping using NVIDIA H200 GPUs (141GB VRAM). Each worker is al...
  [score: 0.7774 | doc: doc-2fd8f544] single GPU. We report Percentile Rank 7 as our primary metric, as discrete medal thresholds are sensitive to noise near ...
  [score: 0.7686 | doc: doc-2fd8f544] splits in addition to train and (the original) test data. While thisnecessitates a degree of human curation during the i...
  [score: 0.7632 | doc: doc-2fd8f544] 2achieved a 91 Percentile Rankthe highest among all evaluated agentsby ensembling an EVA-02 Large (Fang et al., 2024) an...
------------------------


[ANSWER]
AIRA 2 is an autonomous AI‑research agent built around three complementary architectural ideas that together overcome the bottlen

In [16]:
## Document upload example (with docx file):

if __name__ == "__main__":
    file_path = "/home/anish/Documents/Ai agent/data/docx/agentic_ai_automation_guidebook.docx"

    agent = upload_document_pipeline(file_path)

    print("\nDocument Agent Created:")
    print(f"  doc_id     : {agent.doc_id}")
    print(f"  file_name  : {agent.metadata['file_name']}")
    print(f"  summary    : {agent.summary}")
    print(f"  keywords   : {agent.keywords}")
    print(f"  vector DB  : {agent.vector_db}")
    print(f"  embed model: {agent.metadata['embed_model']}")

[INFO] Starting pipeline for 'agentic_ai_automation_guidebook.docx' → doc_id: doc-38599d31
[INFO] Extracting text...
[INFO] Cleaning text...
[INFO] Chunking text...
[INFO] Created 6 chunks.
[INFO] Generating embeddings with Mistral...
[INFO] Storing in Pinecone...


/home/anish/Documents/Ai agent/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Summarizing document...
[INFO] Document split into 2 chunks for summarization.
[INFO] Summarizing chunk 1/2...
[TOKEN] Used 2856 tokens this call | Total this window: 2,856/131,072
[INFO] Summarizing chunk 2/2...
[TOKEN] Used 965 tokens this call | Total this window: 3,821/131,072
[INFO] Merging chunk summaries...
[TOKEN] Used 504 tokens this call | Total this window: 4,325/131,072
[INFO] Agent 'doc-38599d31' saved to system_memory.json
[INFO] Pipeline complete.

Document Agent Created:
  doc_id     : doc-38599d31
  file_name  : agentic_ai_automation_guidebook.docx
  summary    : The document outlines a comprehensive learning path for mastering automation and AI-powered workflows, launching a successful automation agency, and a 4-step sequence for offering high-ticket services. It covers tools, AI agent architecture, and business growth phases.
  keywords   : ['automation', 'AI', 'workflows', 'agency', 'business growth']
  vector DB  : doc-38599d31-index
  embed model: mistral-e